# PolicyProof AI — Stage 11: genuine native AlignTune 40-step QLoRA trial (T4)

**Purpose:** A new, separate, GPU-optional-to-trigger experiment through **`AlignTune wrapper.train()`**, rather than the existing direct-PyTorch 40-step fallback. Official API: [AlignTune SFT](https://aligntune.lexsi.ai/user-guide/sft/). This is a RESEARCH training trial; **it has not been executed or validated against a real T4 by repository CI**.

Use a **fresh Google Colab Python 3 notebook**, select **Runtime → Change runtime type → T4 GPU**, and run cells in order. Your only upload is the verified original `policyproof_curatorkit_train_v1.zip` containing **fictional 78 training examples**. The notebook's only optimizer call is behind `RUN_NATIVE_TRAINING = False`; change it to `True` only after the training-data, 4-bit model and revision checks pass. **Do not run the old Stage 3 smoke or direct-PyTorch training in the same session.**

**Safety and scientific limits:** The dataset is fictional, and 40 optimizer steps do not prove any safety or factual accuracy gain. The original 48-case development evaluation, final 60-case candidate, and 72 challenge cases must **not** be used during training or parameter selection. A model Hub revision SHA is not independently verified loaded weights unless the actual model config exposes the matching SHA. Preserve the original ZIP, training receipt and unedited Colab logs. Tiny Aya's noncommercial CC-BY-NC terms apply.

## 1 — Connect the T4 and clone the pinned project code

No model download or training occurs in this cell. Use a fresh runtime; it refuses an already-loaded `trainer` object to avoid accidental reuse of optimizer state.

In [ ]:
import os, pathlib, subprocess, sys, json
import torch
assert torch.cuda.is_available(), "Select GPU → T4 in Colab Runtime settings."
assert "T4" in torch.cuda.get_device_name(0), "This protected recipe was only checked for Tesla T4."
assert "trainer" not in globals(), "Use a new runtime: existing trainer state may have optimizer steps."
ROOT = pathlib.Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",
        str(ROOT)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
assert (ROOT/"scripts/complete_t4_training.py").is_file()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CODE_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
assert len(CODE_COMMIT) == 40
print("GPU:", torch.cuda.get_device_name(0))
print("Code commit:", CODE_COMMIT)
print("Zero training steps performed.")

## 2 — Install and verify the exact earlier compatible AlignTune environment

This checks pinned **AlignTune 0.1.28**, **TRL 1.7.1** and **BitsAndBytes 0.50.2**; missing or different versions are installed together. Installing libraries can change the live Python environment, so **if Colab asks to restart the runtime, restart and run all cells again**. Nothing here starts training. Global `pip check` warnings from unrelated Colab packages are printed, but core training-import failures are fatal.

In [ ]:
import importlib, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

PINNED = {"aligntune": "0.1.28", "trl": "1.7.1", "bitsandbytes": "0.50.2"}
def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None
before = {name: installed(name) for name in PINNED}
print("Before:", before)
if any(before[name] != expected for name, expected in PINNED.items()):
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--prefer-binary",
        "aligntune==0.1.28", "trl==1.7.1", "bitsandbytes==0.50.2", "jedi"
    ], check=True)
    importlib.invalidate_caches()
after = {name: installed(name) for name in PINNED}
assert after == PINNED, ("Wrong training package versions: " + str(after))
from aligntune.core.backend_factory import create_sft_trainer
from trl import SFTTrainer
from peft import PeftModel
import bitsandbytes
assert callable(create_sft_trainer)
torch.empty(1, device="cuda")
print("TRAINING_ENV_READY: TRUE")
print("Model/GPU training has not started.")

## 3 — Restore the **original** CuratorKIT ZIP and load exactly one approved 4-bit Tiny Aya model

If the verified 78-row dataset is absent, Colab asks you to upload **one** ZIP. The restored dataset must match the original immutable SHA-256. This cell checks Hugging Face model access and records the remote model commit; it also inspects the hash actually visible on the loaded model. The two are **not automatically equivalent**. Loading the base model uses GPU memory but **does not train**.

In [ ]:
from scripts.restore_colab_session import restore_interactive, EXPECTED_SHA
from scripts.check_align_config import check_train_data
trainer, MODEL_SHA, restore_report = restore_interactive(ROOT)
CURATED = ROOT/"artifacts"/"verified_curator_train"
approved = check_train_data(CURATED/"sft_alpaca.jsonl")
assert approved["dataset_sha256"] == EXPECTED_SHA
assert approved["row_count"] == 78
assert restore_report["tokenization"]["checked_rows"] == 78
assert restore_report["4bit_layers"] == 252
assert restore_report["trainable_lora_parameters"] == 3833856
loaded_revision = restore_report["loaded_revision_inspection"]
assert loaded_revision["hub_revision_claim"] == MODEL_SHA
assert int(getattr(getattr(trainer.trainer, "state", None), "global_step", 0)) == 0 if getattr(trainer, "trainer", None) else True
print("Original approved train dataset SHA:", approved["dataset_sha256"])
print("Model Hub revision:", MODEL_SHA)
print("Loaded config revision check:", json.dumps(loaded_revision, indent=2))
print("Base is quantized, LoRA adapter is trainable, 78 examples ready.")
if not loaded_revision["loaded_model_revision_verified"]:
    print("CAUTION: weight revision NOT independently verified; an adapter receipt cannot certify it.")
print("Still ZERO optimizer steps.")

## 4 — Inspect the pretraining state; training remains OFF

The helper changes the AlignTune configuration **only after you opt in to Step 5**. This cell checks the 4-bit base, exactly 288 trainable adapter tensors and 3,833,856 trainable parameters, 0 completed optimizer steps, and the known training-only dataset. It does not create an SFTTrainer or run an optimizer.

In [ ]:
from scripts.t4_no_amp_recovery import audit_training_state
state = audit_training_state(trainer)
assert state["existing_optimizer_steps"] == 0
assert state["base_quantization_4bit"] is True
assert state["trainable_tensor_count"] == 288
assert state["trainable_elements"] == 3833856
assert len(trainer.train_dataset) == 78
PRETRAIN_CHECKED = True
print("Pretrain state:", json.dumps(state, indent=2, default=str))
print("Requested native AlignTune optimizer steps: 40")
print("After you opt in, the helper will verify TRL max_steps=40, fp16=False and bf16=False.")
print("NO optimizer step has occurred.")

## 5 — **Explicit opt-in:** native AlignTune 40-step QLoRA training

**Default OFF.** Change just `RUN_NATIVE_TRAINING = False` to `True`, then run this cell **once**. It adjusts AlignTune's `config.train.max_steps` and its late `train.extra_params["max_steps"]`, verifies the final TRL SFTConfig, repairs only FP32 LoRA parameters, and calls **`trainer.train()`**. If the actual TRL trainer does not complete **exactly 40** optimizer steps, the run is marked failed and no completed adapter ZIP is offered.

Successful local file integrity **still does not prove** external GPU authenticity, factual accuracy or a qualifying hackathon submission. The configured 40-step native run reuses the **same in-memory 78-example training dataset** inspected in Step 4, rather than asking AlignTune to re-download/reprocess it a second time; it does not use development or final cases. If the T4 job fails, share the traceback without tokens and preserve the failure receipt; do NOT relabel a direct-PyTorch fallback as AlignTune.

In [ ]:
RUN_NATIVE_TRAINING = False  # CHANGE ONLY THIS TO True to spend GPU/execute 40 native steps
if RUN_NATIVE_TRAINING:
    assert globals().get("PRETRAIN_CHECKED") is True
    assert getattr(trainer, "model", None) is not None
    from pathlib import Path
    from scripts.complete_t4_training import recover_train_save
    from scripts.verify_training_archive import inspect_training_archive

    existing_zip = Path("/content/policyproof_align_native40.zip")
    native_dir = ROOT / "artifacts/align_qlora_native40"
    if existing_zip.is_symlink() or native_dir.is_symlink():
        raise ValueError("Unsafe native training artifact symlink")
    if existing_zip.exists():
        # Recovery/re-execution safety: reuse and inspect the ORIGINAL result.
        # NEVER enter a second trainer.train() call from a repeated cell.
        assert existing_zip.is_file(), "Existing native output is not a regular file"
        output_zip = existing_zip
        print("Existing native40 ZIP detected: SKIPPING ALL training.")
    else:
        assert not native_dir.exists(), (
            "Native trainer already started but no complete ZIP exists. "
            "Run the READ-ONLY interruption audit; do NOT restart optimizer."
        )
        output_zip = recover_train_save(
            trainer,
            project_dir=str(ROOT),
            model_revision=MODEL_SHA,
            optimizer_steps=40,
            loaded_revision_evidence=loaded_revision,
        )
    cpu_receipt = inspect_training_archive(output_zip)
    assert cpu_receipt["optimizer_steps_self_reported"] == 40
    assert cpu_receipt["native_aligntune_40step_receipt_checked"] is True
    assert cpu_receipt["backend"] == "aligntune_wrapper"
    assert cpu_receipt["expected_tiny_aya_lora_geometry"] is True
    assert cpu_receipt["training_independently_verified"] is False
    assert cpu_receipt["model_improvement_verified"] is False
    print("Native AlignTune 40-step artifact integrity:", json.dumps(
        cpu_receipt, indent=2, ensure_ascii=False,
    ))
    if not cpu_receipt["base_revision_loaded_evidence_verified"]:
        print("BLOCKER: the loaded base model SHA was not independently observable.")
    print("Training result is preserved in Colab:", output_zip)
    print("Run the separate owner-only BACKUP/DOWNLOAD cell below. "
          "Re-executing this cell will NOT train again if a complete ZIP exists.")
else:
    print("Native AlignTune 40-step training remains OFF. No optimizer step executed.")

## 6 — Owner-only write-once private backup and browser download (NO TRAINING)

The previous Stage 5 cell **saves and verifies** native training output, but no longer launches a browser download inside the optimizer action. Use this recovery cell **independently**, as often as needed, without retriggering training.

All controls below default **OFF**. If the real native 40-step ZIP exists, change `SAVE_PRIVATE_DRIVE_BACKUP=True` to copy it into your own **private Google Drive** using Google’s explicit authorization prompt. The backup filename is tied to the original ZIP SHA-256; a matching existing copy is verified and reused, while a conflicting or partial file is **never silently replaced**. The copy is independently read back and its native adapter evidence checked again.

Set `DOWNLOAD_NATIVE40_ZIP=True` to request a fresh browser download **without any optimizer call**. If the download popup fails, rerun **only this recovery cell**, never the training cell. A Google Drive mount is optional and is **not** needed for a browser download.

Preserve the unedited GPU runtime logs separately. Private Drive sync is not a guarantee of durability, and a locally consistent training archive does not authenticate original GPU optimizer execution or establish model improvement. Do not commit or publish adapters without validating license and rights.


In [ ]:
from pathlib import Path
import json
SAVE_PRIVATE_DRIVE_BACKUP = False  # OWNER opt-in; private Drive, NEVER public GitHub
DOWNLOAD_NATIVE40_ZIP = False     # OWNER opt-in; never runs any optimizer
ORIGINAL_NATIVE40_ZIP = Path("/content/policyproof_align_native40.zip")
if SAVE_PRIVATE_DRIVE_BACKUP or DOWNLOAD_NATIVE40_ZIP:
    from scripts.verify_training_archive import inspect_training_archive
    assert ORIGINAL_NATIVE40_ZIP.is_file() and not ORIGINAL_NATIVE40_ZIP.is_symlink(), (
        "Original native40 ZIP absent; do not rerun optimizer to manufacture a replacement."
    )
    verified_native = inspect_training_archive(ORIGINAL_NATIVE40_ZIP)
    assert verified_native["native_aligntune_40step_receipt_checked"] is True
    assert verified_native["optimizer_steps_self_reported"] == 40
    assert verified_native["backend"] == "aligntune_wrapper"
    assert verified_native["training_independently_verified"] is False
    if SAVE_PRIVATE_DRIVE_BACKUP:
        from google.colab import drive
        from scripts.preserve_native40_owner_archive import preserve_native40_archive
        drive.mount("/content/drive")  # secure private Drive owner consent
        private_destination = Path("/content/drive/MyDrive/PolicyProof_Lexsi_Native40")
        backup = preserve_native40_archive(ORIGINAL_NATIVE40_ZIP, private_destination)
        assert backup["archive_sha256"] == verified_native["archive_sha256"]
        assert backup["original_training_gpu_independently_verified"] is False
        print("Private write-once, readback-verified native ZIP:", json.dumps(
            backup, indent=2, ensure_ascii=False))
    if DOWNLOAD_NATIVE40_ZIP:
        from google.colab import files
        files.download(str(ORIGINAL_NATIVE40_ZIP))
else:
    print("No backup or download requested. NO model load, optimizer or cloud write.")
print("Recovery cell finished. No training or model inference performed.")


## 7 — Preserve the complete evidence trail (owner action after real training)

If the run completes, **save the verified original `policyproof_align_native40.zip`, its private backup (if authorized), and the unchanged Colab training log**. The adapter ZIP contains **exactly three files**: `adapter/adapter_model.safetensors`, `adapter/adapter_config.json`, and `run_info.json` with dataset, code, optimizer-step and backend declarations. AlignTune may create additional intermediate checkpoints and tokenizer files in Colab, but they stay outside this portable ZIP. Preserve those separately only if scientifically necessary; do not expose private tokens or logs. The CPU verifier checks format and hash consistency, **not independent GPU execution**.

Next: evaluate both exact-base and adapter on the 48-case development set **without modifying the model**, complete genuine blinded multilingual human review, then only after human approval evaluate the separately frozen 60-case candidate. Record and verify the actual loaded model revision; a remote Hugging Face `model_info().sha` is not proof of loaded weights. Do not publish the adapter as proof of safer answers until independent evidence exists.

## 8 — Read-only interruption check (NO MODEL, NO TRAINING)

If the Stage 11 training cell fails, Colab disconnects, or you find a ZIP but the local receipt says `packaging`, **do not restart the 40-step training**. Run this independent diagnostic cell in the same existing Colab file system. It reads local evidence and, if available, the ZIP and verifies their integrity without modifying or overwriting anything. Preserve both copies and the original log if the status is not `MATCHED_SELF_REPORTED_COMPLETION`.

A verified ZIP/receipt match confirms only internal consistency of the saved artifacts; it is **not independent proof of executed GPU optimization or factual answer quality**. After a complete kernel reset, Colab's temporary files may be unavailable; only an original saved ZIP and logs can be examined.

In [ ]:
from pathlib import Path
import json
from scripts.reconcile_native_training_evidence import inspect_native_training_evidence

EXISTING_TRAINING_DIR = Path("/content/PolicyProof-AI/artifacts/align_qlora_native40")
EXISTING_ADAPTER_ZIP = Path("/content/policyproof_align_native40.zip")
if EXISTING_TRAINING_DIR.is_dir():
    audit = inspect_native_training_evidence(
        EXISTING_TRAINING_DIR,
        EXISTING_ADAPTER_ZIP if EXISTING_ADAPTER_ZIP.is_file() else None,
    )
    print(json.dumps(audit, indent=2, ensure_ascii=False))
    if audit["owner_review_required"]:
        print("EVIDENCE NEEDS OWNER REVIEW: preserve original files; do NOT rerun training.")
else:
    print("No native40 training folder in this Colab runtime. NO training was triggered.")